# Kaggle Homework 4 - Dirks Wright-dirksw

### EDA Notebook

eda notebook: https://github.com/wdirkswright/Advanced-Machine/blob/main/Kaggle_Comp/HW_2_EDA.ipynb

The EDA showed that the target variable is imbalanced. Low irrigation need makes up 369,917 rows, or about 58.7% of the training data, Medium makes up 239,074 rows, or about 37.9%, and High makes up only 21,009 rows, or about 3.3%. Because the High class is much smaller, plain accuracy is not enough by itself, so the HW4 notebook focuses on balanced accuracy, macro F1, and weighted F1. The variables that looked most connected to irrigation need were soil moisture, crop growth stage, mulching, temperature, wind speed, rainfall, and water availability.

## Feature Engineering Notebook

notebook: https://github.com/wdirkswright/Advanced-Machine/blob/main/Kaggle_Comp/HW_4_feature_eng.ipynb

### Goal and Setup

This notebook focused on feature engineering and model ensembling. The goal was to improve performance above 0.96 while still paying attention to the imbalanced High class. I used a stratified 80/20 split to produce training and validation datasets. After feature engineering, the modeling table had 25 numeric features and 16 categorical features.

### Feature Engineering

22 new features were created through feature engineering. The main groups were water availability features, dryness and evaporation interactions, soil condition interactions, categorical combination features, and binned nonlinear versions of moisture, rainfall, and temperature.

A quick HistGradientBoosting comparison showed that the engineered feature set was very close to the original feature set, but it did not clearly beat the original features in that isolated test:

| Feature Set | Accuracy | Balanced Accuracy | Weighted F1 | Macro F1 |
| --- | ---: | ---: | ---: | ---: |
| Original features only | 0.9826 | 0.9653 | 0.9826 | 0.9618 |
| Engineered feature set | 0.9822 | 0.9650 | 0.9822 | 0.9598 |

Even though the engineered features did not improve this one-model comparison, I kept them for the full model comparison because they captured useful domain relationships and gave the ensemble more options.

### Models Tested

This notebook compared eight candidate models across four model families. Logistic Regression and ExtraTrees used one-hot encoded categorical variables, HistGradientBoosting used ordinal encoded categorical variables, and CatBoost handled categorical variables natively.

- Logistic Regression: C=0.5 and C=2.0, both with balanced class weights.
- ExtraTrees: one conservative version with max_features='sqrt' and min_samples_leaf=2, and one more flexible version with max_features=0.7 and min_samples_leaf=1.
- HistGradientBoosting: lr=0.06, leaves=31 and lr=0.04, leaves=63, both with balanced class weights.
- CatBoost: depth=6, lr=0.08 and depth=8, lr=0.05, both using multiclass loss and native categorical handling.

The main model selection metric was balanced accuracy, with weighted F1, macro F1, and accuracy used as supporting metrics. Additionally, cross-validation was used to improve performance and consistency.

### Performance Summary

| Model | CV Balanced Accuracy | CV Weighted F1 | Holdout Balanced Accuracy | Holdout Weighted F1 |
| --- | ---: | ---: | ---: | ---: |
| HistGB lr=0.06 leaves=31 | **0.9647** | 0.9811 | **0.9644** | 0.9823 |
| HistGB lr=0.04 leaves=63 | 0.9628 | 0.9807 | 0.9643 | 0.9828 |
| CatBoost depth=8 lr=0.05 | 0.9618 | **0.9841** | 0.9590 | 0.9839 |
| CatBoost depth=6 lr=0.08 | 0.9617 | 0.9840 | 0.9596 | **0.9839** |
| ExtraTrees 0.7 leaf=1 | 0.9541 | 0.9790 | 0.9580 | 0.9829 |
| ExtraTrees sqrt leaf=2 | 0.9248 | 0.9531 | 0.9401 | 0.9614 |
| Logistic balanced C=0.5 | 0.9015 | 0.8927 | 0.9002 | 0.8898 |
| Logistic balanced C=2.0 | 0.9012 | 0.8932 | 0.9004 | 0.8899 |

The best model by the metric I prioritized was HistGB lr=0.06 leaves=31, with a CV balanced accuracy of 0.9647 and a holdout balanced accuracy of 0.9644. Its holdout weighted F1 was 0.9823. The classification report showed strong performance across all classes, including the minority High class, where precision was 0.90, recall was 0.93, and F1 was 0.91.

CatBoost had the highest weighted F1, but its balanced accuracy was lower than HistGradientBoosting. That matters because weighted F1 can look strong when the majority classes are predicted well, while balanced accuracy gives more direct weight to the smaller High class.

### Useful Features

Permutation importance showed that the strongest predictors were `Crop_Growth_Stage`, `Moisture_Deficit`, `Mulching_Used`, `Temperature_C`, `Wind_Speed_kmh`, `Soil_Moisture`, and `Rainfall_mm`. The most useful engineered feature was `Moisture_Deficit`, which makes sense because it is a direct transformation of soil moisture. Several of the more complicated interaction features had near-zero importance, so the feature engineering helped most when it captured a simple domain signal rather than when it added complexity.

### Ensemble

This notebook built probability-average ensembles from the best model in each family: CatBoost depth=8 lr=0.05, HistGB lr=0.06 leaves=31, ExtraTrees 0.7 leaf=1, and Logistic balanced C=0.5. The equal-average ensemble reached holdout accuracy of 0.9836, balanced accuracy of 0.9637, and weighted F1 of 0.9835. The weighted-average ensemble reached holdout accuracy of 0.9837, balanced accuracy of 0.9636, and weighted F1 of 0.9836.

The ensembles slightly improved accuracy and weighted F1, but they did not beat the best individual HistGradientBoosting model on balanced accuracy. For the final submission file, I refit CatBoost, HistGradientBoosting, and Logistic Regression on the full engineered training data. Trees were left out because of the amount it took to refit, and predict the final probabilities. 

I finally got above 0.96 with a value of 0.96252 and a position on the leaderboard around 2223. This is the highest score I have received so far. However, it would be interesting to see if it was because of feature engineering, the ensemble, cross-validation or HistGB. Regardless, the steps taken in this notebook did lead to improved predictive ability. 

## Things I Could Have Done Differently

The biggest thing I would change is the feature engineering workflow. The quick feature-set test showed that the engineered features were basically tied with the original features. Removing low-importance interaction features might improve generalization and reduce noise.

#### 1. More Targeted Feature Selection
- Keep high-value features like Moisture_Deficit, but remove engineered features with near-zero permutation importance.
- Test feature groups separately so I can see whether water, weather, soil, or categorical interaction features are actually helping.

#### 2. Better Ensemble Tuning
- Try a stacked model with out-of-fold predictions so the ensemble can learn when CatBoost, HistGradientBoosting, and ExtraTrees are each strongest.
- Compare the final ensemble against the best individual HistGradientBoosting model on the Kaggle leaderboard.

#### 3. More Model Search
- Try LightGBM or XGBoost with the engineered features and class weighting, combine these into the ensemble method.
- Tune HistGradientBoosting more carefully because it was the best balanced-accuracy model.
- Continue tuning CatBoost because it had the strongest weighted F1, even though its balanced accuracy was lower.